In [1]:
import gc
import h5py
import numpy as np
import pandas as pd
import anndata as ad
import torch
from pathlib import Path
from prophet import Prophet
from huggingface_hub import hf_hub_download
from prophet.data.dataloader import read_in_priors
from rdkit import Chem
from tqdm import tqdm

UNIPERT_DIR = Path("/storage/pancellflow/data")
OUTPUT_H5AD = Path("/storage/pancellflow/tahoe_prophet_filtered.h5ad")
MATCHED_DRUGS = Path("/storage/pancellflow/drug_dict_with_smiles.csv")

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
_orig_load = torch.load
torch.load = lambda *a, **kw: _orig_load(*a, **{**kw, "weights_only": False})
torch.serialization.add_safe_globals([np.core.multiarray.scalar])
torch.serialization.safe_globals([np.core.multiarray.scalar])

## TAHOE

- Obtain drugs present in tahoe

In [2]:
# Get unique drug names from A549 (all 7 cell lines share the same 367 drugs)
adata_a549 = ad.read_h5ad(UNIPERT_DIR / "tahoe_a549_w_emb.h5ad", backed="r")
tahoe_drugs = [d for d in adata_a549.obs["drug_0"].unique() if d != "control"]
adata_a549.file.close()
print(f"Tahoe drugs: {len(tahoe_drugs)}")

Tahoe drugs: 367


# Load prophet Model

In [8]:
model = Prophet.from_pretrained("base", split="cell_lines", fold=0, seed=110)
model.model.eval()

🔄 Downloading base model from HuggingFace Hub...
   Configuration: split=cell_lines, fold=0, seed=110
Successfully downloaded base model: epoch=29-step=45360.ckpt
  Configuration: split=unseen_cell_lines, fold=0, seed=110
  Model file: /root/.cache/prophet/base_cell_lines_fold0_seed110/datasets--theislab--Prophet/snapshots/7d8b27edeaab117af851e3f5625a018b0f45fa35/base_pretrained/unseen_cell_lines_fold_0/seed_110/epoch=29-step=45360.ckpt
Learning rate set to 1e-05


TransformerPredictor(
  (learnable_embedding): Embedding(1000, 512, max_norm=0.5)
  (embedding_dropout): Dropout(p=0.2, inplace=False)
  (gene_net): Sequential(
    (0): Linear(in_features=1219, out_features=512, bias=True)
    (1): GELU(approximate='none')
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=512, out_features=512, bias=True)
  )
  (drug_net): Sequential(
    (0): Linear(in_features=1219, out_features=512, bias=True)
    (1): GELU(approximate='none')
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=512, out_features=512, bias=True)
  )
  (cl_net): Sequential(
    (0): Linear(in_features=300, out_features=512, bias=True)
    (1): GELU(approximate='none')
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=512, out_features=512, bias=True)
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-7): 8 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQu

# IV_emb

- Obtain IV Embeddings in prophet
- Match drugs by name
- Create matched dict

In [9]:
files = [
    "embeddings/intervention_embeddings/global_iv_scaledv3.csv",
]
paths = [hf_hub_download(repo_id="theislab/Prophet", filename=f, repo_type="dataset") for f in files]
iv_emb1 = read_in_priors(paths)

In [10]:
iv_drugs1 = iv_emb1[iv_emb1["type"] == "drug"].copy()
iv_drugs1 = iv_drugs1[~iv_drugs1.index.duplicated(keep="first")]
iv_name_lookup = {n.strip().lower(): n for n in iv_drugs1.index}

In [11]:
emb_cols = [c for c in iv_drugs1.columns if c not in ("type", "smiles")]
iv_vectors = iv_drugs1[emb_cols]

In [12]:
matched_names, matched_vecs = [], []
for drug in tahoe_drugs:
    drug_norm = drug.strip().lower()
    if drug_norm in iv_name_lookup:
        idx = iv_name_lookup[drug_norm]
        matched_names.append(drug)
        matched_vecs.append(iv_vectors.loc[idx].values.astype(np.float32))

print(f"Matched: {len(matched_names)} / {len(tahoe_drugs)} drugs")

Matched: 182 / 367 drugs


# Prophet Embs

- Obtain prophet embs for matched drugs

In [13]:
# Run through drug_net -> 512-dim
with torch.no_grad():
    embs = model.model.drug_net(
        torch.tensor(np.stack(matched_vecs), dtype=torch.float32)
    ).detach().cpu().numpy()

prophet_emb = {name: embs[i] for i, name in enumerate(matched_names)}
print(f"prophet_emb: {len(prophet_emb)} drugs x {embs.shape[1]}d")

del model, iv_emb1, iv_drugs1
gc.collect()


prophet_emb: 182 drugs x 512d


3799

## Write

- Write updated h5ad needed for cellflow training

In [14]:
CELLLINE_FILES = [
    "tahoe_a549_w_emb.h5ad",
    "tahoe_sw48_w_emb.h5ad",
    "tahoe_snu_1_w_emb.h5ad",
    "tahoe_h4_w_emb.h5ad",
    "tahoe_aspc_1_w_emb.h5ad",
    "tahoe_hop62_w_emb.h5ad",
    "tahoe_panc_1_w_emb.h5ad",
]

prophet_drug_set = set(prophet_emb.keys())
adatas = []

# These two dicts are merged across ALL files
merged_cell_line_emb = {}
merged_drug_emb      = {}
base_uns             = None   # everything else (taken from first file)

for fname in CELLLINE_FILES:
    path = UNIPERT_DIR / fname
    with h5py.File(path, "r") as f:
        a = ad.AnnData(
            X=ad.io.read_elem(f["X"]) if "X" in f else None,
            obs=ad.io.read_elem(f["obs"]),
            obsm=ad.io.read_elem(f["obsm"]),
            uns=ad.io.read_elem(f["uns"]),
        )
        keep = (a.obs["drug_0"] == "control") | a.obs["drug_0"].isin(prophet_drug_set)
        n_kept, n_total = keep.sum(), len(keep)
        a = a[keep].copy()

    # Merge per-file embedding dicts
    merged_cell_line_emb.update(a.uns.get("cell_line_embeddings", {}))
    merged_drug_emb.update(a.uns.get("drug_0_embeddings", {}))

    # Keep non-embedding uns from the first file only
    if base_uns is None:
        base_uns = {k: v for k, v in a.uns.items()
                    if k not in ("cell_line_embeddings", "drug_0_embeddings")}

    a.uns = {}
    print(f"{fname}: {n_kept:,} / {n_total:,} cells kept")
    adatas.append(a)

print(f"\ncell_line_embeddings keys : {list(merged_cell_line_emb.keys())}")
print(f"drug_0_embeddings   keys  : {len(merged_drug_emb)} drugs")

print("\nConcatenating …")
adata = ad.concat(adatas, join="outer")
del adatas
gc.collect()

# Restore fully-merged uns
adata.uns = base_uns
adata.uns["cell_line_embeddings"] = merged_cell_line_emb
adata.uns["drug_0_embeddings"]    = merged_drug_emb
adata.uns["prophet_emb"]          = prophet_emb

adata.obs["control"] = (adata.obs["drug_0"] == "control")
adata.obs["drug"]    = adata.obs["drug_0"]
adata.obs.drop(columns=["drug_0"], inplace=True)

print(f"\nTotal cells : {adata.n_obs:,}")
print(f"Genes       : {adata.n_vars:,}")
print(f"obs cols    : {adata.obs.columns.tolist()}")
print(f"uns keys    : {list(adata.uns.keys())}")

adata.write_h5ad(OUTPUT_H5AD)
print(f"Saved to {OUTPUT_H5AD}")

tahoe_a549_w_emb.h5ad: 1,156,972 / 2,326,890 cells kept
tahoe_sw48_w_emb.h5ad: 422,117 / 849,525 cells kept
tahoe_snu_1_w_emb.h5ad: 586,236 / 1,138,884 cells kept
tahoe_h4_w_emb.h5ad: 484,269 / 952,254 cells kept
tahoe_aspc_1_w_emb.h5ad: 996,554 / 2,026,821 cells kept
tahoe_hop62_w_emb.h5ad: 1,521,395 / 3,009,751 cells kept
tahoe_panc_1_w_emb.h5ad: 1,924,983 / 3,766,070 cells kept

cell_line_embeddings keys : ['CVCL_0023', 'CVCL_1724', 'CVCL_0099', 'CVCL_1239', 'CVCL_0152', 'CVCL_1285', 'CVCL_0480']
drug_0_embeddings   keys  : 368 drugs

Concatenating …

Total cells : 7,092,526
Genes       : 62,710
obs cols    : ['sample', 'species', 'gene_count', 'tscp_count', 'mread_count', 'bc1_wind', 'bc2_wind', 'bc3_wind', 'bc1_well', 'bc2_well', 'bc3_well', 'id', 'drugname_drugconc', 'drug', 'INT_ID', 'NUM.SNPS', 'NUM.READS', 'demuxlet_call', 'BEST.GUESS', 'BEST.LLK', 'NEXT.GUESS', 'NEXT.LLK', 'DIFF.LLK.BEST.NEXT', 'BEST.POSTERIOR', 'SNG.POSTERIOR', 'cell_line', 'SNG.BEST.LLK', 'SNG.NEXT.GUESS', 

In [16]:
import h5py

with h5py.File(OUTPUT_H5AD, "r") as f:
    print(list(f.keys()))                          # top-level groups
    print(list(f["uns"].keys()))                   # check prophet_emb is there
    print(len(f["uns"]["prophet_emb"].keys()))          # check drug names

['X', 'layers', 'obs', 'obsm', 'obsp', 'uns', 'var', 'varm', 'varp']
['AE_pretrained_hvg_var_names', 'X_pca_loadings', 'X_pca_variance', 'X_pca_variance_ratio', 'cell_line_ccle_embedding_dim', 'cell_line_ccle_embeddings', 'cell_line_embedding_dim', 'cell_line_embeddings', 'drug_0_embeddings', 'drug_1_embeddings', 'drug_embedding_dim', 'prophet_emb']
182


In [17]:
adata = ad.read_h5ad(OUTPUT_H5AD, backed="r")
print(adata)
print(adata.obs.columns.tolist())
print(list(adata.uns.keys()))
print(list(adata.uns["prophet_emb"].keys())[:5])

AnnData object with n_obs × n_vars = 7092526 × 62710 backed at '/storage/pancellflow/tahoe_prophet_filtered.h5ad'
    obs: 'sample', 'species', 'gene_count', 'tscp_count', 'mread_count', 'bc1_wind', 'bc2_wind', 'bc3_wind', 'bc1_well', 'bc2_well', 'bc3_well', 'id', 'drugname_drugconc', 'drug', 'INT_ID', 'NUM.SNPS', 'NUM.READS', 'demuxlet_call', 'BEST.GUESS', 'BEST.LLK', 'NEXT.GUESS', 'NEXT.LLK', 'DIFF.LLK.BEST.NEXT', 'BEST.POSTERIOR', 'SNG.POSTERIOR', 'cell_line', 'SNG.BEST.LLK', 'SNG.NEXT.GUESS', 'SNG.NEXT.LLK', 'SNG.ONLY.POSTERIOR', 'DBL.BEST.GUESS', 'DBL.BEST.LLK', 'DIFF.LLK.SNG.DBL', 'sublibrary', 'BARCODE', 'pcnt_mito', 'S_score', 'G2M_score', 'phase', 'cell_line_orig', 'pass_filter', 'cell_name', 'dosage', 'plate', 'pert_compound', 'drug_org', 'drug_1', 'control'
    uns: 'AE_pretrained_hvg_var_names', 'X_pca_loadings', 'X_pca_variance', 'X_pca_variance_ratio', 'cell_line_ccle_embedding_dim', 'cell_line_ccle_embeddings', 'cell_line_embedding_dim', 'cell_line_embeddings', 'drug_0_e